In [1]:
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from pathlib import Path
import shutil
import uuid
import threading
import uvicorn

In [2]:
# ------------------------------------------------------------
# Basic paths
# ------------------------------------------------------------

BASE_DIR = Path.cwd()
IMAGE_DIR = BASE_DIR / "received_images"
MODEL_PATH = BASE_DIR / "local_models" / "sample.glb"

IMAGE_DIR.mkdir(exist_ok=True)

In [3]:
# ------------------------------------------------------------
# FastAPI app
# ------------------------------------------------------------

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ------------------------------------------------------------
# Helper: save uploaded image from Unity
# ------------------------------------------------------------

def save_uploaded_image(file: UploadFile) -> Path:
    """
    Save the image uploaded from Unity Editor.
    This function only saves the image for testing.
    """
    suffix = Path(file.filename).suffix or ".png"
    image_path = IMAGE_DIR / f"{uuid.uuid4()}{suffix}"

    with open(image_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)

    return image_path


# ------------------------------------------------------------
# Helper: send local GLB model to Unity
# ------------------------------------------------------------

def send_local_glb_model():
    """
    Read a local GLB model and send it back to Unity.
    Replace this later with an AI-generated GLB output.
    """
    if not MODEL_PATH.exists():
        raise FileNotFoundError(f"Cannot find model: {MODEL_PATH}")

    return FileResponse(
        path=MODEL_PATH,
        media_type="model/gltf-binary",
        filename="test_model.glb"
    )


# ------------------------------------------------------------
# Unity uploads image here
# ------------------------------------------------------------

@app.post("/generate")
async def generate(file: UploadFile = File(...)):
    """
    Unity sends an image to this endpoint.
    The server saves the image and returns the URL of a local GLB model.
    """
    image_path = save_uploaded_image(file)

    print(f"Received image from Unity: {image_path}")

    return {
        "status": "done",
        "model_url": "http://127.0.0.1:8000/model"
    }


# ------------------------------------------------------------
# Unity downloads GLB model here
# ------------------------------------------------------------

@app.get("/model")
def get_model():
    """
    Unity downloads the local GLB model from this endpoint.
    """
    return send_local_glb_model()


# ------------------------------------------------------------
# Run FastAPI server in a background thread
# This avoids Jupyter's event loop conflict.
# ------------------------------------------------------------

server = None
server_thread = None

def start_server():
    """
    Start the FastAPI server inside JupyterLab without blocking the notebook.
    """
    global server, server_thread

    if server_thread is not None and server_thread.is_alive():
        print("Server is already running at http://127.0.0.1:8000")
        return

    config = uvicorn.Config(
        app,
        host="127.0.0.1",
        port=8000,
        log_level="info"
    )

    server = uvicorn.Server(config)

    server_thread = threading.Thread(
        target=server.run,
        daemon=True
    )

    server_thread.start()

    print("Server started at http://127.0.0.1:8000")


def stop_server():
    """
    Stop the FastAPI server.
    """
    global server

    if server is not None:
        server.should_exit = True
        print("Server stopping...")
    else:
        print("Server is not running.")


start_server()

Server started at http://127.0.0.1:8000


INFO:     Started server process [36264]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


Received image from Unity: C:\Users\Chenkai\OneDrive\PhD-Projects\Collaborative Projects\Jiatong Xia VR 3D Generation Design\received_images\462b0062-46cd-4592-bd0a-e3b45a384d7d.png
INFO:     127.0.0.1:59585 - "POST /generate HTTP/1.1" 200 OK
INFO:     127.0.0.1:59585 - "GET /model HTTP/1.1" 200 OK
